# 07 · Baseline iii — Visual Contrastive Decoding (VCD) on the same detections

**Purpose.** For each image and each distinct candidate label, ask the VLM on the full image whether the object exists, using VCD (logits contrasted against a diffusion-noised copy, α=1, β=0.1, noise step 500). A candidate is kept iff the answer for its label is yes. The same passes also give the plain (non-contrastive) global answer.

| | |
|---|---|
| **Proposal objective** | Baseline iii (decoding-time mitigation) |
| **Inputs** | detection caches |
| **Outputs** | `results/baselines/vcd_*.jsonl`, `results/baselines/vcd_*.json` |
| **Runtime (T4)** | 2 forward passes per (image, label) → fast profile ≈10 min for 450 images |

> Run cells top to bottom. Every expensive stage is **resumable**: if Colab disconnects, reconnect,
> re-run from the top and it continues where it stopped (results are cached on Drive under `results/`).

In [ ]:
# ===================== Bootstrap (identical in every notebook) =====================
import os, sys, subprocess  # paths, sys.path and shell commands (git clone, pip)
IN_COLAB = "google.colab" in sys.modules

# Where the project lives on Google Drive. Upload the whole repo folder there (see docs/COLAB_GUIDE.md),
# or set REPO_URL and it will be cloned into DRIVE_PROJECT on first run.
DRIVE_PROJECT = "/content/drive/MyDrive/detect-verify-decide"
REPO_URL = ""  # e.g. "https://github.com/<you>/detect-verify-decide.git"
BRANCH = "main"  # git branch to take the latest code from (change it if you pushed to another branch)

if IN_COLAB:
    from google.colab import drive  # Colab only: mount Google Drive
    drive.mount("/content/drive")
    if not os.path.isdir(DRIVE_PROJECT) and REPO_URL:
        subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, DRIVE_PROJECT], check=True)
    elif os.path.isdir(os.path.join(DRIVE_PROJECT, ".git")):   # pick up the latest pushed code (never touches your notebooks/results)
        subprocess.run(["git", "-C", DRIVE_PROJECT, "fetch", "origin", BRANCH], check=False)
        subprocess.run(["git", "-C", DRIVE_PROJECT, "checkout", "FETCH_HEAD", "--", "src", "configs", "requirements.txt", "tools", "app"], check=False)
    PROJECT_ROOT = DRIVE_PROJECT
else:
    PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))

assert os.path.isdir(os.path.join(PROJECT_ROOT, "src", "dvd")), f"src/dvd not found under {PROJECT_ROOT}"
os.environ["DVD_ROOT"] = PROJECT_ROOT
sys.path.insert(0, os.path.join(PROJECT_ROOT, "src"))
print("PROJECT_ROOT =", PROJECT_ROOT)

In [ ]:
# Install dependencies (Colab already has torch/torchvision with CUDA). ~1-2 min per fresh runtime.
if IN_COLAB:
    %pip install -q -r "{PROJECT_ROOT}/requirements.txt"

In [ ]:
import json, numpy as np, pandas as pd, matplotlib.pyplot as plt  # json files, numpy arrays, pandas tables, matplotlib plots
from dvd.config import load_config, save_selected, results_dir  # load configs/default.yaml, save tuned values, results folders
from dvd.io_utils import set_seed, save_json, load_json, JsonlCache  # seeding, JSON helpers and the resumable JSONL cache
from dvd import paths  # cache-file path helpers

cfg = load_config()
set_seed(cfg["seed"])
pd.set_option("display.precision", 4)
try:
    import torch  # PyTorch: GPU check and memory statistics
    print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (CPU only - expect very slow runs)")
except ImportError:
    print("torch not installed")
print("data_root   :", cfg["paths"]["data_root"])
print("results_root:", cfg["paths"]["results_root"])
print("tuned params:", cfg["selected"] or "(none yet)")

In [ ]:
from dvd.data.coco import prepare_annotations, fetch_images, CocoData, load_splits  # COCO annotations, image download, splits
prepare_annotations(cfg)   # annotation json only (the 241 MB zip is cached on Drive)
coco = CocoData(cfg)
splits = load_splits(cfg)
HELDOUT, TEST = splits["heldout"], splits["test"]
fetch_images(cfg, HELDOUT + TEST)   # only these images (a few hundred files), not the whole 778 MB val2017 zip
print(f"held-out: {len(HELDOUT)} images | test: {len(TEST)} images")

In [ ]:
from dvd.agents.verifier import build_verifier  # Verifier Agent (frozen Qwen2-VL): factory and crop builder
from dvd.baselines.vcd import run_vcd_stage, diffusion_noise  # baseline (iii): Visual Contrastive Decoding
from dvd.cropping import fit_max_side  # crop-and-zoom / image resizing helpers
from dvd.experiments import SplitData, available_methods, sweep, operating_point, evaluate_row  # methods over identical candidates: sweeps, operating points, evaluation, costs
H = SplitData(cfg, coco, "heldout", HELDOUT); T = SplitData(cfg, coco, "test", TEST)
verifier = build_verifier(cfg)
im = fit_max_side(coco.image(TEST[0]), 448)
fig, ax = plt.subplots(1, 2, figsize=(10, 4)); ax[0].imshow(im); ax[0].set_title("original")
ax[1].imshow(diffusion_noise(im, cfg["vcd"]["noise_step"])); ax[1].set_title(f"noised (step {cfg['vcd']['noise_step']})")
for a in ax: a.axis("off")
plt.show()

In [ ]:
vcd_h = run_vcd_stage(verifier, H.det, HELDOUT, paths.vcd_cache(cfg, "heldout"), cfg, coco.image, "vcd held-out")
vcd_t = run_vcd_stage(verifier, T.det, TEST, paths.vcd_cache(cfg, "test"), cfg, coco.image, "vcd test")

In [ ]:
out = []
for NAME in ["VCD global yes/no", "Plain global yes/no (no contrast)"]:
    mh, mt = available_methods(cfg, H)[NAME], available_methods(cfg, T)[NAME]
    best = operating_point(sweep(NAME, mh, H, cfg))
    r = evaluate_row(mt, T, best, n_boot=1000); r.pop("_masks")
    out.append({"method": NAME, **r})
save_json(out, results_dir(cfg, "baselines") / "vcd_test.json")
pd.DataFrame(out)[["method", "tau", "precision", "recall", "f1", "fp", "fp_per_image"]]